In [14]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler


df = pd.read_csv("./datasets/processed/trek_preprocessed_unscaled.csv")

print("Original shape:", df.shape)

df = (
    df.groupby("Trek")
      .first()
      .reset_index()
)

print("Unique treks:", len(df))

iso_features = [
    "Duration_Days",
    "Max_Altitude_m",
    "Trip_Grade_Ordinal"
]

X = df[iso_features]

iso = IsolationForest(
    contamination=0.10,   # top 10% unusual treks
    random_state=42
)

df["Anomaly"] = iso.fit_predict(X)

df["Anomaly_Score"] = iso.decision_function(X)

score_cols = [
    "Duration_Days",
    "Max_Altitude_m",
    "Trip_Grade_Ordinal",
    "Cost_USD"
]

scaler = MinMaxScaler()

scaled = pd.DataFrame(
    scaler.fit_transform(df[score_cols]),
    columns=score_cols
)

df["Hidden_Gem_Score"] = (
    0.40 * scaled["Max_Altitude_m"]
    + 0.30 * scaled["Duration_Days"]
    + 0.30 * scaled["Trip_Grade_Ordinal"]
) / (scaled["Cost_USD"] + 0.10)


hidden_gems = (
    df[df["Anomaly"] == -1]
    .copy()
)

hidden_gems = hidden_gems.sort_values(
    by="Hidden_Gem_Score",
    ascending=False
)

top10 = hidden_gems[
    [
        "Trek",
        "Cost_USD",
        "Duration_Days",
        "Max_Altitude_m",
        "Trip_Grade_Ordinal",
        "Hidden_Gem_Score"
    ]
].head(10)

print("\nTOP 10 HIDDEN GEMS\n")
print(top10.to_string(index=False))

hidden_gems.to_csv(
    "hidden_gems_ranked.csv",
    index=False
)

print("\nSaved: hidden_gems_ranked.csv")

Original shape: (383, 15)
Unique treks: 68

TOP 10 HIDDEN GEMS

                                      Trek  Cost_USD  Duration_Days  Max_Altitude_m  Trip_Grade_Ordinal  Hidden_Gem_Score
              Tsum Valley and Manaslu Trek    1499.0             27            4460                   6          1.994273
Everest Advanced Base Camp Trek from Tibet    1499.0             18            6340                   5          1.971650
                     Tamang Heritage Trail     750.0              8            2607                   2          0.955741
                     Nepal Experience Tour    1499.0             12            3210                   1          0.616424
                      Short Annapurna Trek     590.0              5            1990                   2          0.579614
                            The Royal Trek    1499.0              9            1730                   4          0.521809
             Nepal Hiking and Culture Tour    1499.0              8            155